In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import cartopy.io.shapereader as shpreader
import pycountry
import openpyxl

In [ ]:
from _helpers import mock_snakemake
snakemake = mock_snakemake(
    "prepare_iron_ore"
)

### File paths

In [ ]:
# Input from retrieve_iron_ore rule
production_fn = snakemake.input.iron_ore
bus_locations_fn = snakemake.input.bus_locations

# Output
production_clustered_fn = snakemake.output.iron_ore

In [ ]:
# Load config
config = snakemake.config

### Get iron ore data

In [ ]:
# Read the CSV data
production = pd.read_csv(production_fn, index_col='ISO_A2')

In [ ]:
production

### Cluster per region

### Cluster countries per region
Map each country in the production data to its region using the region definitions from the config file.

In [ ]:
# Load region definitions from config
regions = config['regions']

# Build a mapping from country name to ISO_A2 code
country_name_to_iso = {}
for country in pycountry.countries:
    country_name_to_iso[country.name] = country.alpha_2
    if hasattr(country, 'official_name'):
        country_name_to_iso[country.official_name] = country.alpha_2

# Build a mapping from ISO_A2 code to region
iso_to_region = {}
for region, countries in regions.items():
    for country in countries:
        for part in country.split('+'):
            name = part.strip()
            code = country_name_to_iso.get(name)
            if code:
                iso_to_region[code] = region

# Map each row in production to its region
production['region'] = production.index.map(lambda iso: iso_to_region.get(iso, 'Other'))


### Aggregate iron ore production by region

In [ ]:
# Group by region: sum production and calculate weighted average price
# We need to do this in two steps since groupby.agg with custom function on single column doesn't have access to other columns

# First, sum the production
production_sum = production.groupby('region')['IronOreProductionMt'].sum()

# Then calculate weighted average cost
production_avg_cost = production.groupby('region').apply(
    lambda group: (group['IronOreCost'].fillna(0) * group['IronOreProductionMt']).sum() / group['IronOreProductionMt'].sum()
    if group['IronOreProductionMt'].sum() > 0 else 0
)

# Combine into a single DataFrame
production_by_region = pd.DataFrame({
    'IronOreProductionMt': production_sum,
    'IronOreCost': production_avg_cost
})

production_by_region

### Save clustered iron ore production data

In [ ]:
production_by_region["IronOreCost"] = production_by_region["IronOreCost"].round(2)
production_by_region.to_csv(production_clustered_fn)